In [1]:
import pandas as pd
import statsmodels.api as sm    
import os
import numpy as np

In [ ]:
base_path = '/myriadfs/projects/ICS_UKB/projects/kasia_project/causal_analysis/'
file_exposure = os.path.join(base_path, 'group_1_clean_filtered_imputed_dataset_df.tsv') # exposure variables 
file_outcome = os.path.join(base_path, 'group_1_outcomes_df_without_qc_df.tsv') # outcome variables 

exposure_data = pd.read_csv(file_exposure, sep='\t') 
outcome_data = pd.read_csv(file_outcome, sep='\t')

print(f'exposure_data shape: {exposure_data.shape}')
print(f'outcome_data shape: {outcome_data.shape}')

exposure_outcome_data = pd.merge(exposure_data, outcome_data, on='eid', how='inner') # merge exposure and outcome data on 'eid' column
print(f'Merged data shape: {exposure_outcome_data.shape}')

In [ ]:
df_model = exposure_outcome_data.copy()

raw_cols = [
    '1289-0.0', '1309-0.0', '1349-0.0',  # Diet
    '864-0.0', '884-0.0', '22036-0.0',   # Activity
    '1050-0.0', '1060-0.0', # Sun
    '1160-0.0',             # Sleep
    '20116-0.0',            # Smoking
    '20117-0.0',            # Alcohol
    '1110-0.0', '2237-0.0', # Electronic
    '2090-0.0', '2040-0.0'  # Mental
]
existing_cols = [i for i in raw_cols if i in df_model.columns] # if column exists in df_model
for col in existing_cols:
    df_model[col] = df_model[col].apply(lambda x: np.nan if x < 0 else x)    

# 1289-0.0: vegetable intake, 1309-0.0: fruit intake
if '1289-0.0' in df_model.columns and '1309-0.0' in df_model.columns: 
    df_model['diet_total'] = (df_model['1289-0.0'] / 3) + df_model['1309-0.0'] # diet_total = vegetable / 3 + fruit    

# 864-0.0: walk activity, 884-0.0: moderate activity
#if '864-0.0' in df_model.columns and '884-0.0' in df_model.columns: 
#    df_model['activity_is_active'] = ((df_model['864-0.0'] >= 5) | (df_model['884-0.0'] >= 5)).astype(int) # activity_is_active = 1 if walk or moderate activity >= 5 days/week, else 0

# 1050-0.0: summer sun exposure, 1060-0.0: winter sun exposure
if '1050-0.0' in df_model.columns and '1060-0.0' in df_model.columns:
    df_model['sun_exposure_avg'] = (df_model['1050-0.0'] + df_model['1060-0.0']) / 2 # sun_exposure_avg = average of summer and winter sun exposure

# 1160-0.0: sleep duration
if '1160-0.0' in df_model.columns:
    df_model['sleep_hours'] = df_model['1160-0.0']
    # sleep_abnormal = 1 if sleep_hours < 7 or sleep_hours > 8, else 0
    df_model['sleep_abnormal'] = df_model['1160-0.0'].apply(lambda x: 0 if 7 <= x <= 8 else 1)

# 22036-0.0: at or above moderate/vigorous/walking recommendation
if '22036-0.0' in df_model.columns:
    df_model['activity_walking_incl_is_active'] = df_model['22036-0.0'] # activity_walking_incl_is_active = 1 if at or above moderate/vigorous/walking recommendation, else 0

#22035-0.0: at or above moderate/vigorous recommendation
#if '22035-0.0' in df_model.columns:
#    df_model['activity_is_active'] = df_model['22035-0.0'] # activity_is_active = 1 if at or above moderate/vigorous recommendation, else 0

# 20116-0.0: smoking status
if '20116-0.0' in df_model.columns:
    df_model['smoking_status'] = df_model['20116-0.0'] # smoking_status

# 20117-0.0: alcohol status
if '20117-0.0' in df_model.columns:
    df_model['alcohol_status'] = df_model['20117-0.0'] # alcohol_status    

# electronic device, mental and BMI (keep same as original)
other_cols = {
    'device_mobile':'1110-0.0', # mobile phone use
    'device_computer':'2237-0.0', # computer use
    'mental_doctor':'2090-0.0', # doctor visit
    'mental_risk':'2040-0.0', # risk taking
    'processed_meat':'1349-0.0', # processed meat 
    'BMI':'21001-0.0' # BMI
}

for new_name, old_name in other_cols.items():
    if old_name in df_model.columns:
        df_model[new_name] = df_model[old_name]

# 'sun_exposure_avg','alcohol_status' missing, so not included in feature_cols for now
base_feature_cols = ['diet_total', 'sleep_hours','activity_walking_incl_is_active',
                'device_computer', 'mental_doctor', 'mental_risk'] # feature columns

print(base_feature_cols)

dummy_feature_names = [] # create dummy variables for categorical variables: device_mobile and smoking_status

if 'device_mobile' in df_model.columns:
    mobile_dummies = pd.get_dummies(df_model['device_mobile'], prefix='mobile', dtype=float)
    mobile_dummies[df_model['device_mobile'].isna()] = np.nan
    if 'mobile_0' in mobile_dummies.columns: # drop the reference category (mobile_0) to avoid multicollinearity
        mobile_dummies = mobile_dummies.drop(columns=['mobile_0'])
df_model = pd.concat([df_model, mobile_dummies], axis=1)
dummy_feature_names.extend(mobile_dummies.columns.tolist())

if 'smoking_status' in df_model.columns:
    smoking_dummies = pd.get_dummies(df_model['smoking_status'], prefix='smoking', dtype=float)
    smoking_dummies[df_model['smoking_status'].isna()] = np.nan
    if 'smoking_0' in smoking_dummies.columns: # drop the reference category (smoking_0) to avoid multicollinearity
        smoking_dummies = smoking_dummies.drop(columns=['smoking_0'])
    df_model = pd.concat([df_model, smoking_dummies], axis=1)
    dummy_feature_names.extend(smoking_dummies.columns.tolist())

if 'processed_meat' in df_model.columns:
    meat_dummies = pd.get_dummies(df_model['processed_meat'], prefix='meat', dtype=float)
    meat_dummies[df_model['processed_meat'].isna()] = np.nan
    if 'meat_0' in meat_dummies.columns: # drop the reference category (meat_0) to avoid multicollinearity
        meat_dummies = meat_dummies.drop(columns=['meat_0'])
    df_model = pd.concat([df_model, meat_dummies], axis=1)
    dummy_feature_names.extend(meat_dummies.columns.tolist())

feature_cols = base_feature_cols + dummy_feature_names
print(feature_cols)

In [10]:
target_col = 'def_CVD_AF_HF_AFTER' # target column
model_cols = ['age_defined_baseline', 'genetic_sex','BMI'] + feature_cols # age, sex, BMI + feature columns
final_cols = [target_col] + model_cols + ['eid'] # final columns for modeling

cols_to_save = [col for col in final_cols if col in df_model.columns]
df_final = df_model[cols_to_save]

my_save_dir = '/home/rmhihbp/my_ukb_thesis/data'
os.makedirs(my_save_dir, exist_ok=True)
save_path = os.path.join(my_save_dir, 'group_1_model_ready.parquet')
df_final.to_parquet(save_path, index=False)